# SKANN-SSL V5 Dataset Generator

**Purpose:** Generate 12,000 synthetic underwater acoustic clips with corrected physics.

**V5 Fixes:**
- ✅ Correct shaft rate ranges (non-overlapping classes)
- ✅ Fixed swell frequency (0.05-0.15 Hz, not 0.5 Hz)
- ✅ Always 3 resonances per clip

**Output:** Saved to Google Drive for persistence.

**Estimated Time:** 12-18 hours on Colab Pro

## 1. Setup & Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create output directory
import os
OUTPUT_DIR = '/content/drive/MyDrive/SKANN_SSL_V5_Dataset'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/waveforms', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/tensors', exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Drive mounted and directories created.")

Mounted at /content/drive
Output directory: /content/drive/MyDrive/SKANN_SSL_V5_Dataset
Drive mounted and directories created.


## 2. Install Dependencies

In [ ]:
# tqdm is pre-installed on Colab, but ensure latest version
!pip install -q tqdm --upgrade

import numpy as np
import pandas as pd
import scipy
from tqdm.auto import tqdm

print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"SciPy: {scipy.__version__}")
print("tqdm: Ready ✅")

NumPy: 2.0.2
Pandas: 2.2.2
SciPy: 1.16.3
tqdm: Ready ✅


## 3. config.py

In [ ]:
%%writefile /content/config.py
"""
SKANN-SSL Stage -1 Configuration
================================
Centralized parameters for synthetic waveform generation.
"""

import numpy as np

# =============================================================================
# SIGNAL PARAMETERS
# =============================================================================
FS = 16000
FS_FULL = 32000
DURATION = 5.0
N_SAMPLES = int(FS * DURATION)

FFT_SIZE = N_SAMPLES
FREQ_RES = FS / FFT_SIZE
NYQUIST = FS // 2

MIN_FREQ = 10.0
MAX_FREQ = NYQUIST

# =============================================================================
# SEA NOISE PARAMETERS
# =============================================================================
SEA_STATES = [0, 1, 3, 6]

TURB_START = 10.0
TURB_END = 37.7
LF_END = 200.0
MF_END = 500.0
HF_END = 100000.0

TURB_ANCHOR_1 = (1.0, 108.0)
TURB_ANCHOR_2 = (40.0, 55.0)

SS3_COEFFS = {
    'turb': (-33.10, 108.0),
    'lf': (9.09, 45.28),
    'mf': (-6.57, 83.15),
    'hf': (-17.11, 113.73),
    'f_t': 30.7
}

# =============================================================================
# SHIP NOISE PARAMETERS - CORRECT NON-OVERLAPPING SHAFT RATES
# =============================================================================
SHIP_SNR_DB = 6.0

VESSEL_CLASSES = {
    'small_craft': {
        'f0_range': (15.0, 30.0),       # 900-1800 RPM - NO OVERLAP
        'n_blades': (2, 3),
        'n_harmonics': 8,
        'n_bpf_harmonics': 4,
        'harmonic_decay': 3.0,
        'bpf_decay': 4.0,
        'broadband_level': 0.3,
        'broadband_rolloff': -3.0,
        'cavitation_peak': 5000.0,
    },
    'fishing_vessel': {
        'f0_range': (4.0, 8.0),          # 240-480 RPM - NO OVERLAP
        'n_blades': (3, 4),
        'n_harmonics': 10,
        'n_bpf_harmonics': 5,
        'harmonic_decay': 3.0,
        'bpf_decay': 4.0,
        'broadband_level': 0.4,
        'broadband_rolloff': -4.0,
        'cavitation_peak': 1500.0,
    },
    'cargo_ship': {
        'f0_range': (1.5, 2.5),          # 90-150 RPM
        'n_blades': (4, 5),
        'n_harmonics': 12,
        'n_bpf_harmonics': 6,
        'harmonic_decay': 3.0,
        'bpf_decay': 4.0,
        'broadband_level': 0.5,
        'broadband_rolloff': -5.0,
        'cavitation_peak': 600.0,
    },
    'tanker': {
        'f0_range': (1.0, 1.5),          # 60-90 RPM
        'n_blades': (4, 6),
        'n_harmonics': 15,
        'n_bpf_harmonics': 8,
        'harmonic_decay': 3.0,
        'bpf_decay': 4.0,
        'broadband_level': 0.6,
        'broadband_rolloff': -6.0,
        'cavitation_peak': 400.0,
    }
}

TONAL_PHASE_JITTER = 0.1
TONAL_FREQ_JITTER = 0.02
TONAL_AMP_JITTER = 0.1

GENERATOR_FREQS = [50.0, 100.0, 150.0, 60.0, 120.0, 180.0]
GENERATOR_PROB = 0.7

# =============================================================================
# CAVITATION PARAMETERS
# =============================================================================
CAVITATION_PROB = 0.5
CAVITATION_INTENSITY_RANGE = (0.2, 0.8)
CAVITATION_MOD_FREQ_RANGE = (5.0, 20.0)
CAVITATION_BANDS = [
    {'f_low': 300, 'f_high': 800, 'weight': 0.3},
    {'f_low': 1500, 'f_high': 3000, 'weight': 0.5},
    {'f_low': 4000, 'f_high': 7000, 'weight': 0.2},
]

# =============================================================================
# OTHER PARAMETERS
# =============================================================================
FLOW_NOISE_ROLLOFF = -5.0
FLOW_NOISE_REF_FREQ = 1000.0
N_SYNTHETIC_CLIPS = 500
RANDOM_SEED = 42
DATA_DIR = './data'
WAVEFORM_DIR = f'{DATA_DIR}/waveforms'
TENSOR_DIR = f'{DATA_DIR}/tensors'
METADATA_FILE = f'{DATA_DIR}/metadata.csv'
NO_VESSEL_CLASS_NAME = 'no_vessel'
NO_VESSEL_REPS_PER_SEA_STATE = 120
NO_VESSEL_START_CLIP_ID = 1920
SEA_STATES_DESIGN = [0, 1, 3, 6]
VESSEL_CLASSES_DESIGN = ['small_craft', 'fishing_vessel', 'cargo_ship', 'tanker']
N_BLADES_OPTIONS = [3, 4, 5]
GEN_FREQ_OPTIONS = [0.0, 50.0]
CAV_INTENSITY_LEVELS = [0.0, 0.3333, 0.6667, 1.0]
FULL_FACTORIAL_REPS = 5
STRUCTURED_DATA_DIR = './structured_dataset'
OLA_OVERLAP = 0.5
OLA_WINDOW = 'hann'
P_REF = 1e-6

Writing /content/config.py


## 4. Knudsen CSV Files

In [ ]:
# SS0CSV.txt
ss0 = """1,99.00395538,39.7516477
2,100.4374438,40.00836255
3,102.2928667,40.04163212
4,104.3819827,40.21134258
5,106.0888414,40.33915676
6,108.3256645,40.49606151
7,110.2280733,40.60354548
8,112.4607787,40.81876433
9,114.2252887,40.86801053
10,116.7238791,41.07269051
11,118.5663349,41.15898956
12,121.2217791,41.31189746
13,123.0537401,41.38372978
14,125.7241395,41.54434574
15,127.7138928,41.66193956
16,130.4611754,41.83385937
17,132.7787425,41.94010114
18,135.3066954,42.04266925
19,137.6208913,42.2473254
20,140.2803525,42.34578657
50,243.649062,45.36754184
100,440.7174343,46.59488591
150,1003.73444,42.82484824
200,2235.896073,37.29228418
250,4753.630071,31.93298111
300,9841.902696,26.92501598
350,23661.90736,20.39435109
400,53097.36712,14.45324173
450,93583.77763,10.33146796"""
with open('/content/SS0CSV.txt', 'w') as f: f.write(ss0)

# SS1CSV.txt
ss1 = """1,99.59172076,49.92489291
2,101.4617969,50.02287162
3,103.3669882,50.16054244
4,105.2120989,50.22201982
5,107.249084,50.38314006
10,117.1948375,51.06566682
20,135.4111152,51.96963235
50,232.2866993,54.9962497
100,526.9266925,55.56382407
150,1191.352598,51.57202592
200,2262.493522,46.77966943
250,4187.954292,42.26099883
300,7958.015998,37.51169489
350,17460.98762,31.58350903
400,32655.01808,27.16266204
450,63494.54321,22.20030803
490,99590.76167,18.85652267"""
with open('/content/SS1CSV.txt', 'w') as f: f.write(ss1)

# SS3CSV.txt
ss3 = """1,99.83624465,62.89864665
2,99.92915206,62.98198667
3,101.7367177,62.97215304
5,105.6235221,63.18225772
10,116.0069732,63.75118167
20,138.4114411,64.74638571
50,232.3802528,66.94207882
100,588.0718451,65.81746914
150,1548.461112,58.85510143
200,3248.528391,53.66957559
250,6482.860731,48.83223615
300,16786.09592,41.87251162
350,27441.11541,37.77869586
400,59803.06785,31.70711019"""
with open('/content/SS3CSV.txt', 'w') as f: f.write(ss3)

# SS6CSV.txt
ss6 = """1,100.7777406,70.93006271
2,102.7678041,71.00672033
5,108.7352725,71.16467301
10,119.1457168,71.39760712
20,140.9035276,71.94019211
50,241.5974404,73.03982331
100,604.5426466,71.77237768
150,1383.505388,66.41130119
200,3191.316569,60.30300331
250,6590.665675,54.89722473
300,14585.03822,49.00529048
350,34695.16065,42.71278105
400,55239.753,39.06731397"""
with open('/content/SS6CSV.txt', 'w') as f: f.write(ss6)

print("✅ All CSV files written")

✅ All CSV files written


## 5. sea_noise.py

In [ ]:
%%writefile /content/sea_noise.py
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional, Union
import warnings
from config import FS, N_SAMPLES, P_REF

TURB_ANCHOR_1 = (10.0, 108.0)
TURB_ANCHOR_2 = (40.0, 55.0)
LF_END = 200.0
MF_END = 500.0
MIN_FREQ = 10.0

class KnudsenModel:
    def __init__(self, csv_path: Optional[str] = None, sea_state: int = 3):
        self.sea_state = sea_state
        self.coefficients = {}
        self.f_t = None
        if csv_path is not None:
            self._fit_from_csv(csv_path)
        else:
            self._set_default_coefficients()

    def _set_default_coefficients(self):
        self.coefficients = {'turb': (-88.01, 108.0), 'lf': (13.30, 36.29), 'mf': (-0.033, 66.96), 'hf': (-16.22, 110.65)}
        self.f_t = 35.1

    def _fit_from_csv(self, csv_path: str):
        df = pd.read_csv(csv_path, header=None, names=['idx', 'f', 'NL']).sort_values('f').reset_index(drop=True)
        f1, NL1 = TURB_ANCHOR_1
        f2, NL2 = TURB_ANCHOR_2
        a_turb = (NL2 - NL1) / (np.log10(f2) - np.log10(f1))
        b_turb = NL1 - a_turb * np.log10(f1)
        lf_mask = (df['f'] >= 100.0) & (df['f'] <= 200.0)
        if lf_mask.sum() >= 2:
            lf_data = df[lf_mask]
            x_lf = np.log10(lf_data['f'].values)
            A_lf = np.vstack([x_lf, np.ones_like(x_lf)]).T
            a_lf, b_lf = np.linalg.lstsq(A_lf, lf_data['NL'].values, rcond=None)[0]
        else:
            a_lf, b_lf = 13.30, 36.29
        self.f_t = 10 ** ((b_turb - b_lf) / (a_lf - a_turb)) if abs(a_lf - a_turb) > 1e-6 else 35.0
        hf_mask = (df['f'] >= 500.0)
        if hf_mask.sum() >= 2:
            hf_data = df[hf_mask]
            x_hf = np.log10(hf_data['f'].values)
            A_hf = np.vstack([x_hf, np.ones_like(x_hf)]).T
            a_hf, b_hf_raw = np.linalg.lstsq(A_hf, hf_data['NL'].values, rcond=None)[0]
        else:
            a_hf, b_hf_raw = -16.22, 110.65
        NL_200 = a_lf * np.log10(200.0) + b_lf
        NL_500 = a_hf * np.log10(500.0) + b_hf_raw
        a_mf = (NL_500 - NL_200) / (np.log10(500.0) - np.log10(200.0))
        b_mf = NL_200 - a_mf * np.log10(200.0)
        b_hf = NL_500 - a_hf * np.log10(500.0)
        self.coefficients = {'turb': (a_turb, b_turb), 'lf': (a_lf, b_lf), 'mf': (a_mf, b_mf), 'hf': (a_hf, b_hf)}

    def nl(self, f: Union[float, np.ndarray]) -> Union[float, np.ndarray]:
        f = np.atleast_1d(np.asarray(f, dtype=float))
        nl = np.zeros_like(f)
        a_lf, b_lf = self.coefficients['lf']
        a_mf, b_mf = self.coefficients['mf']
        a_hf, b_hf = self.coefficients['hf']
        nl_plateau = a_lf * np.log10(self.f_t) + b_lf
        nl[(f >= MIN_FREQ) & (f < self.f_t)] = nl_plateau
        mask_lf = (f >= self.f_t) & (f < LF_END)
        nl[mask_lf] = a_lf * np.log10(f[mask_lf]) + b_lf
        mask_mf = (f >= LF_END) & (f < MF_END)
        nl[mask_mf] = a_mf * np.log10(f[mask_mf]) + b_mf
        nl[f >= MF_END] = a_hf * np.log10(f[f >= MF_END]) + b_hf
        return nl.squeeze()

    def psd(self, f): return 10 ** (self.nl(f) / 10) * 1e-12

class SeaNoiseGenerator:
    def __init__(self, fs: int = FS, n_samples: int = N_SAMPLES):
        self.fs, self.n_samples = fs, n_samples
        self.nyquist, self.freq_res = fs // 2, fs / n_samples
        self.n_pos_freqs = n_samples // 2 + 1
        self.freq_grid = np.fft.rfftfreq(n_samples, 1/fs)
        self.models = {}
        self._load_models()

    def _load_models(self):
        for ss in [0, 1, 3, 6]:
            for d in [Path('/content'), Path.cwd()]:
                p = d / f'SS{ss}CSV.txt'
                if p.exists():
                    self.models[ss] = KnudsenModel(str(p), sea_state=ss)
                    break
            else:
                self.models[ss] = KnudsenModel(sea_state=ss)

    def generate_frame(self, sea_state: int = 3, rng: Optional[np.random.Generator] = None) -> np.ndarray:
        rng = rng or np.random.default_rng()
        model = self.models.get(sea_state, self.models[3])
        freqs = self.freq_grid.copy()
        freqs[0] = 1.0
        psd = model.psd(freqs)
        psd[0] = 0.0
        amplitude = np.sqrt(psd * self.freq_res)
        phases = rng.uniform(0, 2 * np.pi, self.n_pos_freqs)
        phases[0] = 0.0
        if self.n_samples % 2 == 0: phases[-1] = 0.0
        spectrum = amplitude * np.exp(1j * phases)
        return np.fft.irfft(spectrum, n=self.n_samples) * self.n_samples / np.sqrt(2)

Writing /content/sea_noise.py


## 6. ship_noise.py (V5 Fixed)

In [ ]:
%%writefile /content/ship_noise.py
import numpy as np
from scipy import signal as scipy_signal
from dataclasses import dataclass
from typing import Optional, Tuple
from config import FS, N_SAMPLES, NYQUIST, FREQ_RES, VESSEL_CLASSES, CAVITATION_PROB, CAVITATION_INTENSITY_RANGE

FS_BURST = 200000

@dataclass
class VesselParams:
    vessel_class: str = 'cargo_ship'
    shaft_rate: float = 2.0
    n_blades: int = 4
    blade_pass_freq: float = 8.0
    n_harmonics: int = 12
    n_bpf_harmonics: int = 6
    harmonic_decay: float = 3.0
    bpf_decay: float = 4.0
    broadband_level: float = 0.5
    broadband_rolloff: float = -5.0
    has_cavitation: bool = False
    cavitation_intensity: float = 0.0
    cavitation_peak: float = 600.0
    generator_freq: float = 0.0
    equipment_base_freq: float = 0.0
    resonance_freq_1: float = 0.0
    resonance_freq_2: float = 0.0
    resonance_freq_3: float = 0.0
    cavitation_peak_freq: float = 0.0
    n_cavitation_bursts: int = 0

class ShipNoiseGenerator:
    def __init__(self, fs: int = FS, n_samples: int = N_SAMPLES):
        self.fs, self.n_samples = fs, n_samples
        self.duration = n_samples / fs
        self.nyquist, self.freq_res = fs // 2, fs / n_samples
        self.n_pos_freqs = n_samples // 2 + 1
        self.freq_grid = np.fft.rfftfreq(n_samples, 1/fs)

    def create_vessel_params(self, vessel_class: str, rng: Optional[np.random.Generator] = None) -> VesselParams:
        rng = rng or np.random.default_rng()
        vc = VESSEL_CLASSES[vessel_class]
        f0_min, f0_max = vc['f0_range']
        shaft_rate = rng.uniform(f0_min, f0_max)
        n_blades = rng.integers(vc['n_blades'][0], vc['n_blades'][1] + 1)
        has_cav = rng.random() < CAVITATION_PROB
        cav_int = rng.uniform(*CAVITATION_INTENSITY_RANGE) if has_cav else 0.0
        gen_freq = rng.choice([50.0, 60.0]) if rng.random() < 0.7 else 0.0
        return VesselParams(vessel_class=vessel_class, shaft_rate=shaft_rate, n_blades=n_blades,
            blade_pass_freq=shaft_rate * n_blades, n_harmonics=vc['n_harmonics'], n_bpf_harmonics=vc['n_bpf_harmonics'],
            harmonic_decay=vc['harmonic_decay'], bpf_decay=vc['bpf_decay'], broadband_level=vc['broadband_level'],
            broadband_rolloff=vc['broadband_rolloff'], has_cavitation=has_cav, cavitation_intensity=cav_int,
            cavitation_peak=vc['cavitation_peak'], generator_freq=gen_freq)

    def _add_tonal(self, spectrum, freq, amp, rng):
        if freq <= 0 or freq >= self.nyquist: return
        freq += rng.uniform(-0.02, 0.02) * freq
        idx = int(round(freq / self.freq_res))
        if 0 < idx < len(spectrum):
            spectrum[idx] += amp * rng.uniform(0.9, 1.1) * np.exp(1j * rng.uniform(0, 2*np.pi))

    def _gen_tonal_spectrum(self, params, rng):
        spectrum = np.zeros(self.n_pos_freqs, dtype=complex)
        for h in range(1, params.n_harmonics + 1):
            self._add_tonal(spectrum, h * params.shaft_rate, 10 ** (-params.harmonic_decay * (h-1) / 20), rng)
        for h in range(1, params.n_bpf_harmonics + 1):
            self._add_tonal(spectrum, h * params.blade_pass_freq, 1.5 * 10 ** (-params.bpf_decay * (h-1) / 20), rng)
        if params.generator_freq > 0:
            for h in range(1, 4):
                self._add_tonal(spectrum, h * params.generator_freq, 0.3 * 10 ** (-3.0 * (h-1) / 20), rng)
        return spectrum

    def _gen_broadband(self, params, rng):
        freqs = self.freq_grid.copy()
        freqs[0] = 1.0
        amp = params.broadband_level * (freqs / 1000.0) ** (params.broadband_rolloff / 20)
        amp[0] = 0.0
        phases = rng.uniform(0, 2*np.pi, self.n_pos_freqs)
        phases[0] = 0.0
        return amp * np.exp(1j * phases)

    def _add_resonances(self, spectrum, params, rng):
        bands = [(50, 150), (100, 300), (200, 500)]  # V5: Always 3 resonances
        resonances = [rng.uniform(lo, hi) for lo, hi in bands]
        params.resonance_freq_1, params.resonance_freq_2, params.resonance_freq_3 = resonances
        for freq in resonances:
            sigma = freq / rng.uniform(10, 30)
            peak = rng.uniform(0.3, 0.8) * np.exp(-0.5 * ((self.freq_grid - freq) / sigma) ** 2)
            spectrum += peak * np.exp(1j * rng.uniform(0, 2*np.pi))

    def _add_equipment(self, spectrum, params, rng):
        if params.equipment_base_freq == 0.0: return
        if params.equipment_base_freq == -1.0:
            params.equipment_base_freq = 30.0 if params.generator_freq == 60.0 else 25.0
        for h in range(1, rng.integers(4, 8) + 1):
            self._add_tonal(spectrum, h * params.equipment_base_freq, 0.2 * 10 ** (-3.0 * (h-1) / 20), rng)

    def _gen_cavitation(self, params, rng):
        if not params.has_cavitation or params.cavitation_intensity <= 0:
            params.n_cavitation_bursts = params.cavitation_peak_freq = 0
            return np.zeros(self.n_samples)
        n_burst = int(self.duration * FS_BURST)
        burst_sig = np.zeros(n_burst)
        period = 1.0 / params.blade_pass_freq
        n_bursts = int(self.duration / period)
        swell_freq = rng.uniform(0.05, 0.15)  # V5 FIX: Was 0.5 Hz
        swell_phase = rng.uniform(0, 2*np.pi)
        burst_times = []
        for i in range(n_bursts):
            t_nom = i * period
            t_burst = t_nom + 0.1 * np.sin(2*np.pi*swell_freq*t_nom + swell_phase) * period
            if 0 < t_burst < self.duration: burst_times.append(t_burst)
        params.n_cavitation_bursts = len(burst_times)
        params.cavitation_peak_freq = params.cavitation_peak
        for t_burst in burst_times:
            idx = int(t_burst * FS_BURST)
            tau = rng.uniform(50e-6, 200e-6)
            t_local = np.arange(int(tau * FS_BURST * 10)) / FS_BURST
            burst = np.exp(-t_local / tau) * np.sin(2*np.pi * params.cavitation_peak * rng.uniform(0.8, 1.2) * t_local)
            burst *= params.cavitation_intensity * rng.uniform(0.5, 1.5)
            end_idx = min(idx + len(burst), n_burst)
            if end_idx > idx >= 0: burst_sig[idx:end_idx] += burst[:end_idx - idx]
        cav = scipy_signal.decimate(burst_sig, FS_BURST // self.fs, zero_phase=True)
        if len(cav) > self.n_samples: cav = cav[:self.n_samples]
        elif len(cav) < self.n_samples: cav = np.pad(cav, (0, self.n_samples - len(cav)))
        return cav

    def generate(self, vessel_class: Optional[str] = None, params: Optional[VesselParams] = None,
                 rng: Optional[np.random.Generator] = None) -> Tuple[np.ndarray, VesselParams]:
        rng = rng or np.random.default_rng()
        if params is None: params = self.create_vessel_params(vessel_class or 'cargo_ship', rng)
        spectrum = self._gen_tonal_spectrum(params, rng) + self._gen_broadband(params, rng)
        self._add_resonances(spectrum, params, rng)
        self._add_equipment(spectrum, params, rng)
        spectral = np.fft.irfft(spectrum, n=self.n_samples) * self.n_samples / np.sqrt(2)
        return spectral + self._gen_cavitation(params, rng), params

Writing /content/ship_noise.py


## 7. Generator with tqdm Progress Bars

In [ ]:
%%writefile /content/generator_colab.py
import numpy as np
import pandas as pd
from pathlib import Path
import json
from datetime import datetime
from tqdm.auto import tqdm
from sea_noise import SeaNoiseGenerator
from ship_noise import ShipNoiseGenerator

VERSION = "v5.0.0"
FS, N_SAMPLES, P_REF, SNR_DB = 16000, 80000, 1e-6, 6.0
SEA_STATES = [0, 1, 3, 6]
VESSEL_CLASSES = ['small_craft', 'fishing_vessel', 'cargo_ship', 'tanker']
N_BLADES_OPTIONS, GEN_FREQ_OPTIONS = [3, 4, 5], [0, 50]
CAV_LEVELS = [0.0, 0.3333, 0.6667, 1.0]
EQUIP_POLICY = {0: 0.25, 25: 0.75}

def preprocess(w): x = w - np.mean(w); return (x / (np.sqrt(np.mean(x**2)) + 1e-8)).reshape(1, 1, -1).astype(np.float32)

class ColabDatasetGenerator:
    def __init__(self, output_dir, reps=25, no_vessel_reps=600):
        self.output_dir = Path(output_dir)
        self.waveform_dir, self.tensor_dir = self.output_dir / 'waveforms', self.output_dir / 'tensors'
        self.reps, self.no_vessel_reps = reps, no_vessel_reps
        self.sea_gen, self.ship_gen = SeaNoiseGenerator(FS, N_SAMPLES), ShipNoiseGenerator(FS, N_SAMPLES)
        self.checkpoint_file, self.manifest_file = self.output_dir / 'checkpoint.json', self.output_dir / 'master_dataset_manifest.csv'

    def _ensure_dirs(self):
        for d in [self.output_dir, self.waveform_dir, self.tensor_dir]: d.mkdir(parents=True, exist_ok=True)

    def _create_tasks(self):
        tasks = []
        total_vessel = len(SEA_STATES) * len(VESSEL_CLASSES) * len(N_BLADES_OPTIONS) * len(GEN_FREQ_OPTIONS) * len(CAV_LEVELS) * self.reps
        rng = np.random.default_rng(42)
        n_zero = int(total_vessel * EQUIP_POLICY[0])
        equip = np.array([0.0] * n_zero + [-1.0] * (total_vessel - n_zero))
        rng.shuffle(equip)
        clip_id = equip_idx = 0
        for ss in SEA_STATES:
            for vc in VESSEL_CLASSES:
                for nb in N_BLADES_OPTIONS:
                    for gf in GEN_FREQ_OPTIONS:
                        for ci in CAV_LEVELS:
                            for rep in range(self.reps):
                                tasks.append({'clip_id': clip_id, 'type': 'vessel', 'sea_state': ss, 'vessel_class': vc,
                                    'n_blades': nb, 'generator_freq': gf, 'cavitation_intensity': ci, 'equipment_policy': equip[equip_idx], 'repeat_index': rep})
                                clip_id += 1; equip_idx += 1
        for ss in SEA_STATES:
            for rep in range(self.no_vessel_reps):
                tasks.append({'clip_id': clip_id, 'type': 'no_vessel', 'sea_state': ss, 'repeat_index': rep})
                clip_id += 1
        return tasks

    def _gen_vessel(self, task):
        rng = np.random.default_rng(10000 + task['clip_id'])
        sea = self.sea_gen.generate_frame(sea_state=task['sea_state'], rng=rng)
        sea_rms = np.sqrt(np.mean(sea**2))
        params = self.ship_gen.create_vessel_params(task['vessel_class'], rng)
        params.n_blades, params.blade_pass_freq = task['n_blades'], params.shaft_rate * task['n_blades']
        params.generator_freq = task['generator_freq']
        params.has_cavitation = task['cavitation_intensity'] > 0
        params.cavitation_intensity = task['cavitation_intensity']
        params.equipment_base_freq = task['equipment_policy']
        ship_raw, params = self.ship_gen.generate(params=params, rng=rng)
        ship_rms_raw = np.sqrt(np.mean(ship_raw**2))
        target_rms = sea_rms * (10 ** (SNR_DB / 20))
        scale = target_rms / ship_rms_raw if ship_rms_raw > 0 else 1.0
        ship = ship_raw * scale
        combined = sea + ship
        ship_rms, combined_rms = np.sqrt(np.mean(ship**2)), np.sqrt(np.mean(combined**2))
        wf, tf = f'clip_{task["clip_id"]:06d}.npy', f'tensor_{task["clip_id"]:06d}.npy'
        np.save(self.waveform_dir / wf, combined.astype(np.float32))
        np.save(self.tensor_dir / tf, preprocess(combined))
        return {'clip_id': task['clip_id'], 'repeat_index': task['repeat_index'], 'sea_state': task['sea_state'],
            'vessel_class': task['vessel_class'], 'n_blades': task['n_blades'], 'generator_freq': task['generator_freq'],
            'cavitation_intensity': task['cavitation_intensity'], 'shaft_rate': round(params.shaft_rate, 4),
            'blade_pass_freq': round(params.blade_pass_freq, 4), 'has_cavitation': params.has_cavitation,
            'cavitation_peak_freq': params.cavitation_peak_freq, 'n_cavitation_bursts': params.n_cavitation_bursts,
            'equipment_base_freq': params.equipment_base_freq, 'resonance_freq_1': params.resonance_freq_1,
            'resonance_freq_2': params.resonance_freq_2, 'resonance_freq_3': params.resonance_freq_3,
            'sea_rms_pa': sea_rms, 'ship_rms_pa': ship_rms, 'combined_rms_pa': combined_rms, 'scale_factor': scale,
            'sea_spl_db': round(20*np.log10(sea_rms/P_REF+1e-30), 2), 'ship_spl_db': round(20*np.log10(ship_rms/P_REF+1e-30), 2),
            'combined_spl_db': round(20*np.log10(combined_rms/P_REF+1e-30), 2), 'snr_db': round(20*np.log10(ship_rms/sea_rms+1e-30), 4),
            'filename': wf, 'tensor_path': f'tensors/{tf}', 'waveform_path': f'waveforms/{wf}'}

    def _gen_no_vessel(self, task):
        rng = np.random.default_rng(50000 + task['clip_id'])
        sea = self.sea_gen.generate_frame(sea_state=task['sea_state'], rng=rng)
        sea_rms = np.sqrt(np.mean(sea**2))
        wf, tf = f'clip_{task["clip_id"]:06d}.npy', f'tensor_{task["clip_id"]:06d}.npy'
        np.save(self.waveform_dir / wf, sea.astype(np.float32))
        np.save(self.tensor_dir / tf, preprocess(sea))
        return {'clip_id': task['clip_id'], 'repeat_index': task['repeat_index'], 'sea_state': task['sea_state'],
            'vessel_class': 'no_vessel', 'n_blades': 0, 'generator_freq': 0.0, 'cavitation_intensity': 0.0,
            'shaft_rate': 0.0, 'blade_pass_freq': 0.0, 'has_cavitation': False, 'cavitation_peak_freq': 0.0,
            'n_cavitation_bursts': 0, 'equipment_base_freq': 0.0, 'resonance_freq_1': 0.0, 'resonance_freq_2': 0.0,
            'resonance_freq_3': 0.0, 'sea_rms_pa': sea_rms, 'ship_rms_pa': 0.0, 'combined_rms_pa': sea_rms,
            'scale_factor': 0.0, 'sea_spl_db': round(20*np.log10(sea_rms/P_REF+1e-30), 2), 'ship_spl_db': float('-inf'),
            'combined_spl_db': round(20*np.log10(sea_rms/P_REF+1e-30), 2), 'snr_db': float('-inf'),
            'filename': wf, 'tensor_path': f'tensors/{tf}', 'waveform_path': f'waveforms/{wf}'}

    def _load_checkpoint(self):
        if self.checkpoint_file.exists():
            cp = json.load(open(self.checkpoint_file))
            return cp['last_completed'], cp.get('results', [])
        return -1, []

    def _save_checkpoint(self, last, results):
        json.dump({'last_completed': last, 'results': results, 'timestamp': datetime.now().isoformat()}, open(self.checkpoint_file, 'w'))

    def generate(self, checkpoint_interval=500):
        self._ensure_dirs()
        tasks = self._create_tasks()
        total = len(tasks)
        vessel_count = sum(1 for t in tasks if t['type'] == 'vessel')
        print("="*60)
        print(f"🚢 SKANN-SSL {VERSION} Dataset Generator")
        print("="*60)
        print(f"📊 Total: {total:,} clips ({vessel_count:,} vessel + {total-vessel_count:,} no-vessel)")
        print(f"📁 Output: {self.output_dir}")
        print()
        last, results = self._load_checkpoint()
        start = last + 1
        if start > 0: print(f"🔄 Resuming from {start:,}/{total:,}\n")
        pbar = tqdm(tasks[start:], initial=start, total=total, desc="🎵 Generating", unit="clip",
            bar_format='{desc}: {percentage:3.1f}%|{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]')
        ckpt_count = 0
        for i, task in enumerate(pbar, start=start):
            results.append(self._gen_vessel(task) if task['type'] == 'vessel' else self._gen_no_vessel(task))
            ckpt_count += 1
            pbar.set_postfix_str(f"{task.get('vessel_class', 'no_vessel')[:10]}")
            if ckpt_count >= checkpoint_interval:
                self._save_checkpoint(i, results)
                pbar.write(f"💾 Checkpoint @ {i+1:,}")
                ckpt_count = 0
        pbar.close()
        print("\n📝 Saving manifest...")
        df = pd.DataFrame(results).sort_values('clip_id').reset_index(drop=True)
        df.to_csv(self.manifest_file, index=False)
        if self.checkpoint_file.exists(): self.checkpoint_file.unlink()
        json.dump({'dataset_version': VERSION, 'total_clips': len(df), 'timestamp': datetime.now().isoformat()},
            open(self.output_dir / 'dataset_fingerprint.json', 'w'), indent=2)
        print(f"\n✅ Complete! {len(df):,} clips saved to {self.manifest_file}")
        return df

Writing /content/generator_colab.py


## 8. Verify Setup

In [ ]:
import os, sys
sys.path.insert(0, '/content')

files = ['config.py', 'sea_noise.py', 'ship_noise.py', 'generator_colab.py', 'SS0CSV.txt', 'SS1CSV.txt', 'SS3CSV.txt', 'SS6CSV.txt']
print("Checking files...")
for f in files:
    print(f"  {'✅' if os.path.exists(f'/content/{f}') else '❌'} {f}")

print("\nTesting generators...")
import numpy as np
from sea_noise import SeaNoiseGenerator
from ship_noise import ShipNoiseGenerator
from config import VESSEL_CLASSES

sea_gen = SeaNoiseGenerator()
ship_gen = ShipNoiseGenerator()
print(f"  Sea noise: {sea_gen.generate_frame(3).shape}")
print("\n  Shaft rate ranges (V5 - NON-OVERLAPPING):")
for vc in ['tanker', 'cargo_ship', 'fishing_vessel', 'small_craft']:
    r = VESSEL_CLASSES[vc]['f0_range']
    _, p = ship_gen.generate(vessel_class=vc)
    print(f"    {vc:15s}: {r[0]:5.1f}-{r[1]:5.1f} Hz, actual={p.shaft_rate:.2f} Hz")

print("\n✅ Ready to generate!")

Checking files...
  ✅ config.py
  ✅ sea_noise.py
  ✅ ship_noise.py
  ✅ generator_colab.py
  ✅ SS0CSV.txt
  ✅ SS1CSV.txt
  ✅ SS3CSV.txt
  ✅ SS6CSV.txt

Testing generators...
  Sea noise: (80000,)

  Shaft rate ranges (V5 - NON-OVERLAPPING):
    tanker         :   1.0-  1.5 Hz, actual=1.36 Hz
    cargo_ship     :   1.5-  2.5 Hz, actual=2.38 Hz
    fishing_vessel :   4.0-  8.0 Hz, actual=5.94 Hz
    small_craft    :  15.0- 30.0 Hz, actual=26.50 Hz

✅ Ready to generate!


## 9. Generate Dataset 🚀

**⏱️ Takes 12-18 hours** | **💾 Checkpoints every 500 clips** | **🔄 Resume-safe**

In [ ]:
from generator_colab import ColabDatasetGenerator

OUTPUT_DIR = '/content/drive/MyDrive/SKANN_SSL_V5_Dataset'

generator = ColabDatasetGenerator(
    output_dir=OUTPUT_DIR,
    reps=25,           # 9,600 vessel clips
    no_vessel_reps=600 # 2,400 no-vessel clips
)

df = generator.generate(checkpoint_interval=500)

🚢 SKANN-SSL v5.0.0 Dataset Generator
📊 Total: 12,000 clips (9,600 vessel + 2,400 no-vessel)
📁 Output: /content/drive/MyDrive/SKANN_SSL_V5_Dataset



🎵 Generating: 0.0%|                              | 0/12000 [00:00<?, ?clip/s]

💾 Checkpoint @ 500
💾 Checkpoint @ 1,000
💾 Checkpoint @ 1,500
💾 Checkpoint @ 2,000
💾 Checkpoint @ 2,500
💾 Checkpoint @ 3,000
💾 Checkpoint @ 3,500
💾 Checkpoint @ 4,000
💾 Checkpoint @ 4,500
💾 Checkpoint @ 5,000
💾 Checkpoint @ 5,500
💾 Checkpoint @ 6,000
💾 Checkpoint @ 6,500
💾 Checkpoint @ 7,000
💾 Checkpoint @ 7,500
💾 Checkpoint @ 8,000
💾 Checkpoint @ 8,500
💾 Checkpoint @ 9,000
💾 Checkpoint @ 9,500
💾 Checkpoint @ 10,000
💾 Checkpoint @ 10,500
💾 Checkpoint @ 11,000
💾 Checkpoint @ 11,500
💾 Checkpoint @ 12,000

📝 Saving manifest...

✅ Complete! 12,000 clips saved to /content/drive/MyDrive/SKANN_SSL_V5_Dataset/master_dataset_manifest.csv


## 10. Validate Dataset

In [ ]:
import pandas as pd
df = pd.read_csv(f'{OUTPUT_DIR}/master_dataset_manifest.csv')

print(f"Total: {len(df):,} clips")
print(f"\nClass distribution:")
for vc, n in df['vessel_class'].value_counts().items(): print(f"  {vc}: {n:,}")

print(f"\nShaft rate ranges:")
vessels = df[df['vessel_class'] != 'no_vessel']
for vc in ['tanker', 'cargo_ship', 'fishing_vessel', 'small_craft']:
    d = vessels[vessels['vessel_class'] == vc]['shaft_rate']
    print(f"  {vc:15s}: {d.min():.2f} - {d.max():.2f} Hz")

fish = vessels[vessels['vessel_class'] == 'fishing_vessel']['shaft_rate']
small = vessels[vessels['vessel_class'] == 'small_craft']['shaft_rate']
print(f"\n{'✅ NO OVERLAP' if fish.max() < small.min() else '❌ OVERLAP'} (gap: {small.min() - fish.max():.1f} Hz)")

Total: 12,000 clips

Class distribution:
  small_craft: 2,400
  fishing_vessel: 2,400
  cargo_ship: 2,400
  tanker: 2,400
  no_vessel: 2,400

Shaft rate ranges:
  tanker         : 1.00 - 1.50 Hz
  cargo_ship     : 1.50 - 2.50 Hz
  fishing_vessel : 4.00 - 8.00 Hz
  small_craft    : 15.01 - 30.00 Hz

✅ NO OVERLAP (gap: 7.0 Hz)


## 11. Download Manifest (Optional)

In [ ]:
from google.colab import files
files.download(f'{OUTPUT_DIR}/master_dataset_manifest.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 11: Generate Pairing Manifest for SSL Training
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from tqdm.auto import tqdm

OUTPUT_DIR = '/content/drive/MyDrive/SKANN_SSL_V5_Dataset'
K_VESSEL, K_NO_VESSEL = 6, 3

VESSEL_HIERARCHY = [
    "n_blades", "sea_state", "cavitation_peak_freq", "shaft_rate",
    "generator_freq", "cavitation_intensity", "equipment_base_freq",
    "resonance_freq_1", "has_cavitation", "resonance_freq_2",
    "n_cavitation_bursts", "resonance_freq_3"
]

NO_VESSEL_PAIRING = {0: [6, 3, 1], 1: [6, 0, 3], 3: [0, 1, 6], 6: [0, 1, 3]}

def generate_vessel_pairs(df, vessel_class, k=K_VESSEL):
    class_df = df[df['vessel_class'] == vessel_class].copy()
    if len(class_df) == 0: return []
    raw_weights = np.array([(len(VESSEL_HIERARCHY) - i) ** 1.5 for i in range(len(VESSEL_HIERARCHY))])
    weights = raw_weights / raw_weights.sum()
    class_norm = class_df.copy()
    available_features = [f for f in VESSEL_HIERARCHY if f in class_df.columns]
    for col in available_features:
        class_norm[col] = (class_df[col] - class_df[col].mean()) / (class_df[col].std() + 1e-8)
    feats = class_norm[available_features].values
    clip_ids = class_df['clip_id'].values
    weighted_feats = feats * np.sqrt(weights[:len(available_features)])
    dist_matrix = cdist(weighted_feats, weighted_feats, metric='euclidean')
    pairing_data = []
    for i in range(len(clip_ids)):
        distances = dist_matrix[i].copy()
        distances[i] = -np.inf
        top_k = np.argsort(distances)[::-1][:k]
        pairing_data.append({"anchor_clip_id": int(clip_ids[i]),
            "partner_clip_ids": "|".join(map(str, clip_ids[top_k])), "vessel_class": vessel_class})
    return pairing_data

def generate_no_vessel_pairs(df, k=K_NO_VESSEL):
    nv_df = df[df['vessel_class'] == 'no_vessel'].copy()
    if len(nv_df) == 0: return []
    lookup = {ss: {r['repeat_index']: r['clip_id'] for _, r in nv_df[nv_df['sea_state'] == ss].iterrows()}
              for ss in NO_VESSEL_PAIRING}
    pairing_data = []
    for _, row in nv_df.iterrows():
        partners = [lookup[ps].get(row['repeat_index'], nv_df[nv_df['sea_state'] == ps].iloc[0]['clip_id'])
                    for ps in NO_VESSEL_PAIRING.get(row['sea_state'], [])[:k] if ps in lookup]
        if partners:
            pairing_data.append({"anchor_clip_id": int(row['clip_id']),
                "partner_clip_ids": "|".join(map(str, partners)), "vessel_class": "no_vessel"})
    return pairing_data

# Run
print("🔗 Generating pairing manifest...")
df = pd.read_csv(f'{OUTPUT_DIR}/master_dataset_manifest.csv')
if 'has_cavitation' in df.columns: df['has_cavitation'] = df['has_cavitation'].astype(float)

all_pairs = []
for vc in tqdm(['small_craft', 'fishing_vessel', 'cargo_ship', 'tanker']):
    all_pairs.extend(generate_vessel_pairs(df, vc))
all_pairs.extend(generate_no_vessel_pairs(df))

pairing_df = pd.DataFrame(all_pairs)
pairing_df.to_csv(f'{OUTPUT_DIR}/pairing_manifest.csv', index=False)
print(f"✅ Saved {len(pairing_df):,} pairings to {OUTPUT_DIR}/pairing_manifest.csv")

🔗 Generating pairing manifest...


  0%|          | 0/4 [00:00<?, ?it/s]

✅ Saved 12,000 pairings to /content/drive/MyDrive/SKANN_SSL_V5_Dataset/pairing_manifest.csv
